# Week 6 Demo / Assignment Template  
## Land-Cover Classification from Earth Observation Data  
### CSV path with optional Google Earth Engine data pull

This notebook supports two ways to complete the same GeoAI workflow.

## Required / beginner path: CSV workflow

Use instructor-provided CSV files:

1. `landcover_training_points_demo_v2.csv` — labeled training points.
2. `landcover_prediction_grid_demo_v2.csv` — unlabeled grid points for a map-like prediction plot.

This path works in **Google Colab**, **Orange**, or another table-based tool.

## Optional / advanced path: Earth Engine workflow

Students with Google Earth Engine access may authenticate inside Colab and pull a small Sentinel-2 prediction grid for the Hartford, Connecticut / Connecticut River corridor.

If Earth Engine access does not work, use the CSV path. You can still complete the assignment.

## Study area

All students use the same general study area:

**Hartford, Connecticut / Connecticut River corridor**

Approximate bounding box:

- West: `-72.78`
- South: `41.68`
- East: `-72.52`
- North: `41.88`

This area includes water, vegetation, built-up areas, and open/bare land, making it useful for a simple land-cover classification demo.

## Step 1: Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

# Part A: Required CSV workflow

Use this section if you are completing the assignment with Colab, Orange, or another table-based tool.

The same CSV files can be loaded into Orange.

## Step 2: Load the training CSV

The training CSV contains labeled land-cover samples.

Each row has satellite bands, spectral indices, coordinates, and a land-cover class label.

In [ ]:
# Option 1: Use course GitHub raw links when provided.
training_csv_url = ""  # paste training CSV raw GitHub link here, if available

if training_csv_url:
    training_df = pd.read_csv(training_csv_url)
else:
    # Option 2: manually upload this file to Colab.
    training_df = pd.read_csv("landcover_training_points_demo_v2.csv")

training_df.head()

## Step 3: Inspect the training data

In [ ]:
print("Rows and columns:", training_df.shape)
print("\nColumn names:")
print(training_df.columns.tolist())

print("\nClass counts:")
print(training_df["land_cover"].value_counts())

In [ ]:
training_df.describe()

## Step 4: Map-like scatter plot of labeled training points

This is not a full raster map. It is a simple spatial plot.

- x-axis: longitude
- y-axis: latitude
- color: land-cover class

**Orange equivalent:** Use **Scatter Plot** with longitude as X, latitude as Y, and `land_cover` as color.

In [ ]:
plt.figure(figsize=(8, 6))

for label in training_df["land_cover"].unique():
    subset = training_df[training_df["land_cover"] == label]
    plt.scatter(
        subset["longitude"],
        subset["latitude"],
        label=label,
        s=18,
        alpha=0.75
    )

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Labeled Training Points")
plt.legend()
plt.show()

## Step 5: Select features and target

The target label is `land_cover`.

For this basic demo, use satellite bands and indices as input features.  
Do not use latitude and longitude as model features; use them only for plotting.

In [ ]:
target_column = "land_cover"

feature_columns = [
    "blue",
    "green",
    "red",
    "nir",
    "swir",
    "ndvi",
    "ndwi",
    "ndbi"
]

X = training_df[feature_columns]
y = training_df[target_column]

print("Features:", feature_columns)
print("Target:", target_column)

## Step 6: Split into training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Step 7: Train and compare models

We compare three simple classifiers:

1. Decision Tree
2. Random Forest
3. Logistic Regression

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42, max_depth=4)
tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

forest_model = RandomForestClassifier(random_state=42, n_estimators=100)
forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)
forest_accuracy = accuracy_score(y_test, forest_predictions)

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_accuracy = accuracy_score(y_test, logistic_predictions)

results = pd.DataFrame([
    {"Model": "Decision Tree", "Accuracy": tree_accuracy},
    {"Model": "Random Forest", "Accuracy": forest_accuracy},
    {"Model": "Logistic Regression", "Accuracy": logistic_accuracy}
])

results.round(3)

## Step 8: Confusion matrix and classification report

A confusion matrix helps show which classes are confused.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    forest_predictions,
    xticks_rotation=45
)

plt.title("Random Forest Confusion Matrix")
plt.show()

print(classification_report(y_test, forest_predictions))

## Step 9: Feature importance

This helps identify which bands or indices were useful for the Random Forest model.

In [ ]:
importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": forest_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance

In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(importance["Feature"], importance["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.show()

## Step 10: Load prediction grid CSV

The prediction grid CSV contains unlabeled points from the study area.

The trained model will predict a land-cover class for each grid point.

**Orange equivalent:** Load the prediction grid CSV, connect it to **Predictions** along with the trained model, then use **Scatter Plot**.

In [ ]:
prediction_grid_csv_url = ""  # paste prediction grid CSV raw GitHub link here, if available

if prediction_grid_csv_url:
    grid_df = pd.read_csv(prediction_grid_csv_url)
else:
    # Option 2: manually upload this file to Colab.
    grid_df = pd.read_csv("landcover_prediction_grid_demo_v2.csv")

grid_df.head()

## Step 11: Predict land cover for grid points

In [ ]:
grid_X = grid_df[feature_columns]

grid_df["predicted_land_cover"] = forest_model.predict(grid_X)

grid_df.head()

## Step 12: Map-like scatter plot of predicted land cover

This plot uses:

- x-axis: longitude
- y-axis: latitude
- color: predicted land-cover class

This is a map-like spatial prediction, not a formal GIS raster.

In [ ]:
plt.figure(figsize=(9, 7))

for label in sorted(grid_df["predicted_land_cover"].unique()):
    subset = grid_df[grid_df["predicted_land_cover"] == label]
    plt.scatter(
        subset["longitude"],
        subset["latitude"],
        label=label,
        s=8,
        alpha=0.85
    )

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Map-like Prediction Plot: Predicted Land Cover")
plt.legend(markerscale=2)
plt.show()

# Part B: Optional Earth Engine data pull

Use this section only if you have access to Google Earth Engine.

This optional section pulls Sentinel-2 imagery for the Hartford study area, creates a composite, calculates indices, samples a small grid of points, and then applies the model trained above.

If authentication does not work, skip this section and use the CSV prediction grid.

## Step B1: Authenticate Earth Engine

Students need:

1. A Google account.
2. Earth Engine access enabled for that account.
3. Permission to authenticate Colab.

If you cannot access Earth Engine, skip this optional section.

In [ ]:
# Optional Earth Engine setup.
# Run this only if you want to pull data from Google Earth Engine.

# !pip -q install geemap

# import ee
# import geemap

# ee.Authenticate()
# ee.Initialize()

## Step B2: Define the Hartford study area and Sentinel-2 composite

This uses Sentinel-2 surface reflectance imagery.

The code below is optional and may need Earth Engine authentication.

In [ ]:
# Optional Earth Engine data pull.
# Uncomment and run after successful Earth Engine authentication.

# study_area = ee.Geometry.Rectangle([-72.78, 41.68, -72.52, 41.88])

# def mask_s2_clouds(image):
#     # Basic cloud mask using QA60 bits.
#     qa = image.select("QA60")
#     cloud_bit_mask = 1 << 10
#     cirrus_bit_mask = 1 << 11
#     mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
#         qa.bitwiseAnd(cirrus_bit_mask).eq(0)
#     )
#     return image.updateMask(mask).divide(10000)

# s2 = (
#     ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
#     .filterBounds(study_area)
#     .filterDate("2023-06-01", "2023-09-30")
#     .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
#     .map(mask_s2_clouds)
# )

# composite = s2.median().clip(study_area)

# image = composite.select(
#     ["B2", "B3", "B4", "B8", "B11"],
#     ["blue", "green", "red", "nir", "swir"]
# )

# ndvi = image.normalizedDifference(["nir", "red"]).rename("ndvi")
# ndwi = image.normalizedDifference(["green", "nir"]).rename("ndwi")
# ndbi = image.normalizedDifference(["swir", "nir"]).rename("ndbi")

# feature_image = image.addBands([ndvi, ndwi, ndbi])

## Step B3: Sample Earth Engine grid points

This creates a small table of pixel samples from the Sentinel-2 image.

The table is then converted into a pandas DataFrame so we can use the same model trained above.

In [ ]:
# Optional: sample points from Earth Engine and convert to pandas.
# Keep the number of pixels small for a beginner notebook.

# sample = feature_image.sample(
#     region=study_area,
#     scale=30,
#     numPixels=2500,
#     seed=42,
#     geometries=True
# )

# features = sample.getInfo()["features"]

# ee_rows = []
# for f in features:
#     props = f["properties"]
#     lon, lat = f["geometry"]["coordinates"]
#     row = {
#         "longitude": lon,
#         "latitude": lat,
#         "blue": props.get("blue"),
#         "green": props.get("green"),
#         "red": props.get("red"),
#         "nir": props.get("nir"),
#         "swir": props.get("swir"),
#         "ndvi": props.get("ndvi"),
#         "ndwi": props.get("ndwi"),
#         "ndbi": props.get("ndbi"),
#     }
#     ee_rows.append(row)

# ee_grid_df = pd.DataFrame(ee_rows).dropna()
# ee_grid_df.head()

## Step B4: Predict and plot Earth Engine grid points

This applies the Random Forest model trained from the labeled CSV to the Earth Engine grid points.

This is a simple teaching example. For a formal mapping project, training samples and prediction imagery should be carefully matched, validated, and documented.

In [ ]:
# Optional: predict Earth Engine sampled grid points.
# Run only after creating ee_grid_df above.

# ee_grid_X = ee_grid_df[feature_columns]
# ee_grid_df["predicted_land_cover"] = forest_model.predict(ee_grid_X)

# plt.figure(figsize=(9, 7))

# for label in sorted(ee_grid_df["predicted_land_cover"].unique()):
#     subset = ee_grid_df[ee_grid_df["predicted_land_cover"] == label]
#     plt.scatter(
#         subset["longitude"],
#         subset["latitude"],
#         label=label,
#         s=8,
#         alpha=0.85
#     )

# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.title("Optional EE Pull: Map-like Prediction Plot")
# plt.legend(markerscale=2)
# plt.show()

# Reflection questions

Answer these in your assignment report.

1. Which data path did you use: CSV only, Earth Engine optional pull, Orange, or another tool?
2. Which input features did you use?
3. Which model performed best?
4. Which classes were easiest to classify?
5. Which features were most useful?
6. What does the map-like scatter plot show?
7. How is a map-like scatter plot different from a full raster land-cover map?
8. What limitations should be considered before using this result for environmental management?

# Final takeaway

This assignment shows that GeoAI can move from Earth observation data to environmental information.

The CSV path teaches the classification workflow in a beginner-friendly way.  
The optional Earth Engine path shows how satellite data can be pulled directly into Colab when Earth Engine access is available.  
The scatter plot prediction gives a simple spatial output without requiring students to create a full georeferenced raster.